# Perbandingan Metode Optimasi Hiperparameter dan Interpretasi SHAP pada LightGBM untuk Prediksi Kepatuhan Pengobatan Pasien Penyakit Tidak Menular

**Ihya' Nashirudin Abrar** — Magister Informatika, Universitas Ahmad Dahlan (2026)

Notebook ini memuat **seluruh eksperimen tesis dari awal sampai akhir** dalam satu alur yang dapat dijalankan ulang:

| Bagian | Isi | Rujukan di tesis |
|---|---|---|
| 1 | Setup, pustaka, dan *random seed* | Subbab 3.2.1 |
| 2 | Pemuatan data dan eksplorasi | Subbab 3.2.2, 4.1 |
| 3 | Pra-pemrosesan (deduplikasi, *encoding*, *feature engineering*, *stratified split*) | Subbab 3.3.2, 4.1 |
| 4 | Studi pendahuluan: lima algoritma *boosting* (justifikasi pemilihan LightGBM) | Lampiran 2 |
| 5 | **Tahap 1** — Optimasi hiperparameter (6 metode) + uji Friedman dan Nemenyi | Subbab 3.3.3, 4.2 |
| 6 | Pelatihan model final | Subbab 4.2 |
| 7 | **Tahap 2** — Interpretasi SHAP empat level | Subbab 3.3.4, 4.3 |
| 8 | Evaluasi kinerja model final (per kelas, IK 95% *bootstrap*, Youden eksploratif) | Subbab 3.3.5, 4.4 |
| 9 | Analisis sensitivitas (pembobotan kelas, ablasi fitur unit klaim, *baseline* vs final) | Subbab 4.5 |
| 10 | Ringkasan | Bab 5 |

Seluruh komponen stokastik memakai `random_state = 42`. Waktu eksekusi penuh sekitar 35–45 menit (CPU), terutama pada Tahap 1.

## 1. Setup dan Pustaka

In [1]:
import warnings, time, json, os
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, GridSearchCV,
                                      RandomizedSearchCV)
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix, roc_curve, auc,
                             classification_report)
from scipy import stats
from scipy.stats import randint, uniform, loguniform

from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import shap

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from skopt import BayesSearchCV
from skopt.space import Integer, Real
import scikit_posthocs as sp

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'Final Prepared Dataset - Diabetes and Hypertension Data.xlsx'
FIG = str(ROOT / 'figures')
OUT = str(ROOT / 'outputs')
os.makedirs(FIG, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

# konsistensi tema visual mengikuti gaya paper
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 11,
    'axes.titleweight': 'bold', 'axes.titlesize': 13,
    'figure.facecolor': 'white', 'savefig.bbox': 'tight',
})
PALETTE = ['#2E5A87', '#C0504D', '#4F8A5B', '#E39A2C', '#7B5EA7']
import sys, sklearn, lightgbm, xgboost, catboost, scipy
print('Pustaka siap. Versi kunci:')
print(f'  python={sys.version.split()[0]} sklearn={sklearn.__version__} lightgbm={lightgbm.__version__} '
      f'optuna={optuna.__version__} shap={shap.__version__} scipy={scipy.__version__}')
print(f'  xgboost={xgboost.__version__} catboost={catboost.__version__}')

Pustaka siap. Versi kunci:
  python=3.11.1 sklearn=1.8.0 lightgbm=4.6.0 optuna=4.8.0 shap=0.49.1 scipy=1.15.3
  xgboost=1.6.2 catboost=1.2.10


## 2. Pemuatan Data dan Eksplorasi

Dataset publik *Final Prepared Dataset — Diabetes and Hypertension Data* (Kanyongo, 2024; Kanyongo et al., 2025b), lisensi CC0 1.0, DOI [10.17632/zkp7sbbx64.2](https://doi.org/10.17632/zkp7sbbx64.2). Target **ADHERENCE** ditentukan dari frekuensi pengambilan ulang (*refill*) obat bulanan dalam 12 bulan: **ADHERENT** = 9–12 kali (≥ 75%), **NON-ADHERENT** = 1–8 kali.

In [2]:
df = pd.read_excel(DATA_PATH)
print('Ukuran data mentah :', df.shape)
print('Kolom              :', list(df.columns))
print('\nDistribusi target ADHERENCE:')
print(df['ADHERENCE'].value_counts())
print('\nProporsi (%):')
print((df['ADHERENCE'].value_counts(normalize=True) * 100).round(2))
df.head()

Ukuran data mentah : (24084, 12)
Kolom              : ['ADHERENCE', 'AGE', 'ANNUALCONTRIBUTION', 'ANNUALCLAIMAMOUNT', 'UNITSTOTAL', 'GENDER_M', 'SCHEMETYPE_MEDIUM', 'SCHEMETYPE_PREMIUM', 'DIAGNOSIS_HYPERTENSION', 'COVERTYPE_STANDARD', 'COMORBIDITY_NO_COMORBIDITY', 'COMPLICATIONDEVELOPMENT_NO_COMPLICATION']

Distribusi target ADHERENCE:
ADHERENCE
NON-ADHERENT    14415
ADHERENT         9669
Name: count, dtype: int64

Proporsi (%):
ADHERENCE
NON-ADHERENT    59.85
ADHERENT        40.15
Name: proportion, dtype: float64


,ADHERENCE,AGE,ANNUALCONTRIBUTION,ANNUALCLAIMAMOUNT,UNITSTOTAL,GENDER_M,SCHEMETYPE_MEDIUM,SCHEMETYPE_PREMIUM,DIAGNOSIS_HYPERTENSION,COVERTYPE_STANDARD,COMORBIDITY_NO_COMORBIDITY,COMPLICATIONDEVELOPMENT_NO_COMPLICATION
0,NON-ADHERENT,24,690384.0,10400.00,60.0,False,False,False,False,True,True,True
1,NON-ADHERENT,53,2718696.0,306366.53,1380.0,True,True,False,False,True,True,True
2,NON-ADHERENT,46,808284.0,77480.00,600.0,True,False,False,False,True,False,True
3,NON-ADHERENT,76,808284.0,198800.58,840.0,True,False,False,False,True,False,True
4,ADHERENT,69,808284.0,116872.04,720.0,False,False,False,False,True,False,True


In [3]:
# --- Gambar: distribusi target ---
counts = df['ADHERENCE'].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
bars = ax[0].bar(counts.index, counts.values, color=[PALETTE[1], PALETTE[0]], edgecolor='black', width=0.6)
ax[0].set_title('Distribusi Kelas Target')
ax[0].set_ylabel('Jumlah Pasien')
for b, v in zip(bars, counts.values):
    ax[0].text(b.get_x()+b.get_width()/2, v+150, f'{v:,}\n({v/counts.sum()*100:.1f}%)',
               ha='center', va='bottom', fontsize=10, fontweight='bold')
ax[0].set_ylim(0, counts.max()*1.18)
ax[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=90,
          colors=[PALETTE[1], PALETTE[0]], wedgeprops=dict(edgecolor='white', linewidth=2),
          textprops=dict(fontsize=11))
ax[1].set_title('Proporsi Kelas Target')
plt.tight_layout(); plt.savefig(f'{FIG}/fig01_distribusi_target.png'); plt.show()
imbalance = counts.max()/counts.min()
print(f'Rasio ketidakseimbangan = {imbalance:.2f} : 1')

Rasio ketidakseimbangan = 1.49 : 1


## 3. Pra-Pemrosesan Data

1. Deduplikasi baris eksak.
2. *Encoding* target `ADHERENT = 1`, `NON-ADHERENT = 0`; fitur boolean dikonversi ke integer 1/0.
3. *Feature engineering* per baris (tanpa statistik agregat, sehingga aman dilakukan sebelum *split*): `LOG_CLAIM`, `LOG_UNITS`, `LOG_CONTRIB` (log1p), `CLAIM_PER_UNIT`, `CLAIM_RATIO` (rasio + log1p), dan `UNITS_PER_AGE` → **17 fitur**.
4. *Stratified split* 80:20.

Tidak dilakukan normalisasi (LightGBM berbasis pohon), tidak ada pembobotan kelas, dan tidak ada SMOTE pada model utama. Pengaruh pembobotan kelas diuji terpisah pada Bagian 9.

In [4]:
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(f'Baris duplikat dihapus : {n_dup}  ->  sisa {df.shape[0]} rekaman unik')

df_p = df.copy()
df_p['ADHERENCE'] = (df_p['ADHERENCE'] == 'ADHERENT').astype(int)
for c in df_p.columns:
    if df_p[c].dtype == 'bool':
        df_p[c] = df_p[c].astype(int)

y = df_p['ADHERENCE']
X = df_p.drop(columns=['ADHERENCE']).copy()

# --- feature engineering (Paper 2) ---
X['LOG_CLAIM']      = np.log1p(X['ANNUALCLAIMAMOUNT'])
X['LOG_UNITS']      = np.log1p(X['UNITSTOTAL'])
X['LOG_CONTRIB']    = np.log1p(X['ANNUALCONTRIBUTION'])
X['CLAIM_PER_UNIT'] = np.log1p(X['ANNUALCLAIMAMOUNT'] / (X['UNITSTOTAL'] + 1))
X['CLAIM_RATIO']    = np.log1p(X['ANNUALCLAIMAMOUNT'] / (X['ANNUALCONTRIBUTION'] + 1))
X['UNITS_PER_AGE']  = X['UNITSTOTAL'] / (X['AGE'] + 1)

print(f'Jumlah fitur setelah engineering : {X.shape[1]}')
print('Fitur:', list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f'\nTrain: {X_train.shape[0]} sampel | Test: {X_test.shape[0]} sampel')
print('Proporsi kelas train:', dict(y_train.value_counts(normalize=True).round(3)))

Baris duplikat dihapus : 13  ->  sisa 24071 rekaman unik
Jumlah fitur setelah engineering : 17
Fitur: ['AGE', 'ANNUALCONTRIBUTION', 'ANNUALCLAIMAMOUNT', 'UNITSTOTAL', 'GENDER_M', 'SCHEMETYPE_MEDIUM', 'SCHEMETYPE_PREMIUM', 'DIAGNOSIS_HYPERTENSION', 'COVERTYPE_STANDARD', 'COMORBIDITY_NO_COMORBIDITY', 'COMPLICATIONDEVELOPMENT_NO_COMPLICATION', 'LOG_CLAIM', 'LOG_UNITS', 'LOG_CONTRIB', 'CLAIM_PER_UNIT', 'CLAIM_RATIO', 'UNITS_PER_AGE']

Train: 19256 sampel | Test: 4815 sampel
Proporsi kelas train: {0: 0.598, 1: 0.402}


In [5]:
# --- Gambar: korelasi antar fitur ---
plt.figure(figsize=(11, 9))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=False,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.7, 'label': 'Koefisien Korelasi'})
plt.title('Matriks Korelasi Antar Fitur (17 Fitur)')
plt.tight_layout(); plt.savefig(f'{FIG}/fig02_korelasi_fitur.png'); plt.show()

## 4. Studi Pendahuluan — Justifikasi Pemilihan LightGBM

Sebagai studi pendahuluan (bukan bagian inti tesis), lima algoritma boosting dievaluasi singkat untuk memastikan LightGBM berada pada tingkatan performa terbaik sehingga layak dijadikan fokus. Evaluasi memakai **10-fold stratified CV** dengan *class weighting* native masing-masing algoritma. Hasilnya (Wilcoxon) menunjukkan LightGBM setara statistik dengan performa terbaik sekaligus jauh lebih efisien.

> Bagian ini **bukan** analisis utama tesis. Bagian ini hanya menjadi dasar pemilihan LightGBM dan dilaporkan pada publikasi terpisah. Berbeda dengan model utama, studi pendahuluan ini memakai pembobotan kelas bawaan masing-masing algoritma.

In [6]:
CV10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

# bobot sampel balanced untuk AdaBoost & GradientBoosting
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()  # scale_pos_weight utk XGBoost

def make_models():
    return {
        'XGBoost': XGBClassifier(
            n_estimators=100, learning_rate=0.1, scale_pos_weight=pos_weight,
            random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss',
            use_label_encoder=False, verbosity=0),
        'AdaBoost': AdaBoostClassifier(
            n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE),
        'LightGBM': LGBMClassifier(
            n_estimators=100, learning_rate=0.1, class_weight='balanced',
            random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
        'CatBoost': CatBoostClassifier(
            n_estimators=100, learning_rate=0.1, auto_class_weights='Balanced',
            random_state=RANDOM_STATE, verbose=0),
    }

NEEDS_SW = {'AdaBoost', 'Gradient Boosting'}  # pakai sample_weight balanced

def cv_boosting(name, model):
    rows = []
    for k, (tr, te) in enumerate(CV10.split(X_train, y_train), 1):
        Xtr, Xte = X_train.iloc[tr], X_train.iloc[te]
        ytr, yte = y_train.iloc[tr], y_train.iloc[te]
        mdl = make_models()[name]
        if name in NEEDS_SW:
            sw = compute_sample_weight('balanced', ytr)
            mdl.fit(Xtr, ytr, sample_weight=sw)
        else:
            mdl.fit(Xtr, ytr)
        yp = mdl.predict(Xte); pp = mdl.predict_proba(Xte)[:, 1]
        rows.append(dict(fold=k,
            accuracy=accuracy_score(yte, yp),
            precision=precision_score(yte, yp, average='weighted'),
            recall=recall_score(yte, yp, average='weighted'),
            f1=f1_score(yte, yp, average='weighted'),
            mcc=matthews_corrcoef(yte, yp),
            auc=roc_auc_score(yte, pp)))
    return pd.DataFrame(rows)

boost_folds = {}
print('Menjalankan 10-fold CV untuk 5 algoritma boosting...\n')
for name, mdl in make_models().items():
    t0 = time.time()
    fdf = cv_boosting(name, mdl)
    boost_folds[name] = fdf
    print(f'  {name:20s} acc={fdf.accuracy.mean():.4f}±{fdf.accuracy.std():.4f} '
          f'auc={fdf.auc.mean():.4f}  mcc={fdf.mcc.mean():.4f}  ({time.time()-t0:.1f}s)')

Menjalankan 10-fold CV untuk 5 algoritma boosting...



  XGBoost              acc=0.8184±0.0076 auc=0.9029  mcc=0.6551  (25.8s)


  AdaBoost             acc=0.7873±0.0075 auc=0.8725  mcc=0.6222  (31.3s)


  Gradient Boosting    acc=0.8116±0.0068 auc=0.8992  mcc=0.6492  (61.4s)


  LightGBM             acc=0.8173±0.0088 auc=0.9028  mcc=0.6508  (4.2s)


  CatBoost             acc=0.8123±0.0064 auc=0.9005  mcc=0.6492  (10.2s)


In [7]:
# --- Ringkasan metrik studi pendahuluan ---
metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'mcc', 'auc']
boost_summary = pd.DataFrame({
    name: {mk: fdf[mk].mean() for mk in metric_keys} for name, fdf in boost_folds.items()
}).T.round(4)
boost_summary_std = pd.DataFrame({
    name: {mk: fdf[mk].std() for mk in metric_keys} for name, fdf in boost_folds.items()
}).T.round(4)
print('Rata-rata metrik 10-fold CV (studi pendahuluan):')
print(boost_summary.to_string())
best_boost = boost_summary['accuracy'].idxmax()
print(f'\n>> Algoritma terbaik (akurasi CV tertinggi): {best_boost}')
boost_summary.to_csv(f'{OUT}/studi_pendahuluan_boosting_summary.csv')

Rata-rata metrik 10-fold CV (studi pendahuluan):


                   accuracy  precision  recall      f1     mcc     auc
XGBoost              0.8184     0.8423  0.8184  0.8201  0.6551  0.9029
AdaBoost             0.7873     0.8366  0.7873  0.7882  0.6222  0.8725
Gradient Boosting    0.8116     0.8420  0.8116  0.8133  0.6492  0.8992
LightGBM             0.8173     0.8395  0.8173  0.8191  0.6508  0.9028
CatBoost             0.8123     0.8416  0.8123  0.8139  0.6492  0.9005

>> Algoritma terbaik (akurasi CV tertinggi): XGBoost


In [8]:
# --- Gambar: bar metrik overall 5 algoritma ---
fig, ax = plt.subplots(figsize=(12, 5.5))
algos = list(boost_summary.index)
xpos = np.arange(len(metric_keys)); w = 0.15
for i, algo in enumerate(algos):
    vals = boost_summary.loc[algo, metric_keys].values
    errs = boost_summary_std.loc[algo, metric_keys].values
    ax.bar(xpos + i*w, vals, w, yerr=errs, capsize=2.5, label=algo,
           color=PALETTE[i], edgecolor='black', linewidth=0.5)
ax.set_xticks(xpos + w*2); ax.set_xticklabels([m.upper() for m in metric_keys])
ax.set_ylabel('Skor'); ax.set_ylim(0.5, 1.0)
ax.set_title('Perbandingan Metrik 10-Fold CV — 5 Algoritma Boosting')
ax.legend(ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.08), frameon=False)
plt.tight_layout(); plt.savefig(f'{FIG}/fig03_boosting_metrik.png'); plt.show()

In [9]:
# --- Gambar: boxplot akurasi per-fold ---
fig, ax = plt.subplots(figsize=(9.5, 5.2))
box_data = [boost_folds[a]['accuracy'].values for a in algos]
bp = ax.boxplot(box_data, labels=algos, patch_artist=True, widths=0.55,
                medianprops=dict(color='black', linewidth=1.5))
for patch, c in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(c); patch.set_alpha(0.75)
ax.set_ylabel('Akurasi'); ax.set_title('Distribusi Akurasi 10-Fold CV per Algoritma')
plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(f'{FIG}/fig04_boosting_boxplot.png'); plt.show()

In [10]:
# --- Fit holdout: ROC + confusion matrix semua algoritma ---
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
roc_ax = axes[0, 0]
boost_holdout = {}
for i, (name, _) in enumerate(make_models().items()):
    mdl = make_models()[name]
    if name in NEEDS_SW:
        mdl.fit(X_train, y_train, sample_weight=sw_train)
    else:
        mdl.fit(X_train, y_train)
    yp = mdl.predict(X_test); pp = mdl.predict_proba(X_test)[:, 1]
    boost_holdout[name] = dict(
        accuracy=accuracy_score(y_test, yp), auc=roc_auc_score(y_test, pp),
        mcc=matthews_corrcoef(y_test, yp), model=mdl)
    fpr, tpr, _ = roc_curve(y_test, pp)
    roc_ax.plot(fpr, tpr, color=PALETTE[i], lw=2,
                label=f'{name} (AUC={auc(fpr,tpr):.3f})')
roc_ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.6)
roc_ax.set_xlabel('False Positive Rate'); roc_ax.set_ylabel('True Positive Rate')
roc_ax.set_title('Kurva ROC (Holdout)'); roc_ax.legend(fontsize=8, loc='lower right')

for i, name in enumerate(make_models().keys()):
    r, c = divmod(i+1, 3)
    cm = confusion_matrix(y_test, boost_holdout[name]['model'].predict(X_test))
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=axes[r, c],
                xticklabels=['NON-ADH','ADH'], yticklabels=['NON-ADH','ADH'])
    axes[r, c].set_title(f'{name}\nacc={boost_holdout[name]["accuracy"]:.3f}', fontsize=10)
    axes[r, c].set_xlabel('Prediksi'); axes[r, c].set_ylabel('Aktual')
plt.tight_layout(); plt.savefig(f'{FIG}/fig05_boosting_roc_cm.png'); plt.show()

In [11]:
# --- Wilcoxon signed-rank test antar algoritma (akurasi per-fold) ---
pairs = algos
wilcox_p = pd.DataFrame(np.ones((len(pairs), len(pairs))), index=pairs, columns=pairs)
for i in range(len(pairs)):
    for j in range(len(pairs)):
        if i != j:
            a = boost_folds[pairs[i]]['accuracy'].values
            b = boost_folds[pairs[j]]['accuracy'].values
            try:
                stat, p = stats.wilcoxon(a, b)
            except ValueError:
                p = 1.0
            wilcox_p.iloc[i, j] = p
plt.figure(figsize=(7.5, 6))
sns.heatmap(wilcox_p, annot=True, fmt='.3f', cmap='RdYlGn_r', center=0.05,
            vmin=0, vmax=0.2, linewidths=0.5, square=True,
            cbar_kws={'label': 'p-value'})
plt.title("Wilcoxon Signed-Rank Test — p-value Antar Algoritma\n(p < 0.05 = beda signifikan)")
plt.tight_layout(); plt.savefig(f'{FIG}/fig06_wilcoxon.png'); plt.show()
print('Wilcoxon p-value (baris vs kolom):')
print(wilcox_p.round(4).to_string())

Wilcoxon p-value (baris vs kolom):
                   XGBoost  AdaBoost  Gradient Boosting  LightGBM  CatBoost
XGBoost             1.0000     0.002             0.0039    0.5410    0.0020
AdaBoost            0.0020     1.000             0.0020    0.0020    0.0020
Gradient Boosting   0.0039     0.002             1.0000    0.0254    0.5391
LightGBM            0.5410     0.002             0.0254    1.0000    0.0098
CatBoost            0.0020     0.002             0.5391    0.0098    1.0000
